In [ ]:
import ipywidgets as widgets
from IPython.display import display
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets
from PIL import Image
import io

# 1. Initialize widgets
uploader = widgets.FileUpload(accept='image/*', multiple=False)
out = widgets.Output()  # This is the 'container' for your image

image_grey = None  # To store the result globally

def on_upload_change(change):
    global image_grey
    
    # We use 'with out' so display() knows exactly where to put the image
    with out:
        clear_output()  # Removes the previous image if you upload a new one
        
        # Check if a file was actually uploaded (handles the tuple in v8+)
        if not change['new']:
            return
            
        # Get the first file from the upload tuple
        file_info = change['new'][0]
        
        # Convert the 'content' bytes into a PIL Image
        input_image = Image.open(io.BytesIO(file_info['content']))
        
        # Process: Convert to Greyscale
        image_grey = input_image.convert('L')
        
        # Display the result
        print(f"Showing: {file_info['name']}")
        display(image_grey)

# 2. Set the observer to watch for the 'value' change
uploader.observe(on_upload_change, names='value')

# 3. Display the button and the output area
display(uploader, out)

In [ ]:
image_grey_np = np.array(image_grey)
print(f"Image shape: {image_grey_np.shape}")
print("Pixels: ")
print(image_grey_np)

In [ ]:
"""Convolution is the process that applies a filter (called a kernel) to an image 
by sliding a small matrix of weights over every pixel and its neighbors, 
multiplying the kernel values with the corresponding pixel values, summing the result, 
and using that value as the new pixel intensity. 
In this system, the image is first converted into a NumPy array so it can be treated as numerical data instead of a visual object, 
and the kernel defines the type of transformation (such as blur, sharpen, or edge enhancement). 
The convolution function then repeatedly extracts small regions of the image, applies the kernel to each region, 
and builds a new output image of the same size, 
effectively transforming the original image based on the chosen filter while preserving its structure.




The system works by converting the uploaded image into a grayscale NumPy array 
so that each pixel becomes a numerical value that can be mathematically manipulated.
The convolution function then uses padding to preserve the original image size, 
ensuring that edge pixels are still processed correctly instead of being lost.
Each kernel acts as a predefined transformation rule, 
where different patterns of weights control how much influence neighboring pixels have on the final result.
The interactive buttons and caching system (precomputed dictionary) 
improve performance by storing previously computed results so filters like blur 
and sharpen can be applied instantly without recalculating everything each time.
This combination of image-to-data conversion, kernel-based transformation, 
and interactive UI elements forms a complete image processing pipeline that allows real-time visual manipulation of digital images.

"""


import numpy as np  # numerical operations (arrays, math for image processing)
import ipywidgets as widgets  # interactive UI widgets for Jupyter (buttons, upload, etc.)
from PIL import Image  # image handling (open, convert, manipulate images)
from IPython.display import display, clear_output  # display images + clear previous outputs
import io  # used to convert uploaded file bytes into an image


# -----------------------------
# Upload widget setup
# -----------------------------
uploader = widgets.FileUpload(accept='image/*', multiple=False)  # file upload button (images only)
output = widgets.Output()  # output box where results (images/text) will be shown

image_grey = None  # stores the uploaded grayscale image globally
precomputed = {}  # dictionary to store processed results (blur, sharpen, etc.)


# -----------------------------
# Handle image upload
# -----------------------------
def on_upload_change(change):  # function triggered when user uploads an image
    global image_grey, precomputed  # allow modifying global variables

    precomputed = {}  # reset cached results whenever a new image is uploaded

    with output:  # ensure all prints/images go into output box
        clear_output()  # remove previous image/output

        if not change['new']:  # if no file uploaded, exit function
            return

        file_info = change['new'][0]  # get uploaded file data (first file only)

        img = Image.open(io.BytesIO(file_info['content']))  # convert uploaded bytes → image

        image_grey = img.convert('L')  # convert image to grayscale ('L' mode = luminance)

        print("Uploaded:", file_info['name'])  # show filename
        display(image_grey)  # display grayscale image


# connect upload widget to handler function
uploader.observe(on_upload_change, names='value')



# Convolution function (same-size output)
def convolve2d_same(image, kernel):  # applies filter (kernel) to image
    h, w = image.shape  # image height and width
    kh, kw = kernel.shape  # kernel height and width

    pad_h, pad_w = kh // 2, kw // 2  # padding size (half kernel size)

    padded = np.pad(  # add border around image so output stays same size
        image,
        ((pad_h, pad_h), (pad_w, pad_w)),  # padding for rows and columns
        mode='edge'  # extend edge pixels outward (prevents black borders)
    )

    output = np.zeros((h, w), dtype=np.float32)  # empty output image (same size as input)

    for i in range(h):  # loop over image rows
        for j in range(w):  # loop over image columns
            output[i, j] = np.sum(  # compute convolution result
                padded[i:i+kh, j:j+kw] * kernel  # element-wise multiply + sum
            )

    return np.clip(output, 0, 255)  # ensure pixel values stay valid (0–255)



# Blur kernel (average filter)
blur_kernel = np.array([
    [1/9, 1/9, 1/9],  # equal weight blur
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9]
])


# Sharpen kernel (edge enhancement)
sharpen_kernel = np.array([
    [0, -1, 0],  # subtract neighbors
    [-1, 5, -1],  # boost center pixel
    [0, -1, 0]
])



# Extra blur kernel (Gaussian blur style)
extra_blur_kernel = np.array([
    [1, 4, 6, 4, 1],    # Gaussian-like weights
    [4,16,24,16, 4],
    [6,24,36,24, 6],
    [4,16,24,16, 4],
    [1, 4, 6, 4, 1]
]) / 256  # normalize so brightness stays consistent



# Precompute results (speed optimization)
def precompute():  # computes all filters at once
    global precomputed  # use global cache

    if image_grey is None:  # if no image uploaded yet
        return

    img = np.array(image_grey)  # convert image → NumPy array

    precomputed['blur'] = convolve2d_same(img, blur_kernel)  # store blur result
    precomputed['sharpen'] = convolve2d_same(img, sharpen_kernel)  # store sharpen result
    precomputed['extra_blur'] = convolve2d_same(img, extra_blur_kernel)  # store extra blur result


# Display result helper
def show_result(key, title):  # shows a processed image
    if image_grey is None:  # check if image exists
        with output:
            clear_output()
            print("Upload image first!")
        return

    if key not in precomputed:  # if not computed yet
        precompute()  # compute filters

    result = Image.fromarray(precomputed[key].astype(np.uint8))  # convert array → image

    with output:  # display inside output box
        clear_output()
        print(title)
        display(result)


# Buttons (UI controls)
blur_btn = widgets.Button(description="Blur")  # blur button
sharpen_btn = widgets.Button(description="Sharpen")  # sharpen button
extra_blur_btn = widgets.Button(description="Extra Blur")  # stronger blur button


# connect buttons to actions
blur_btn.on_click(lambda b: show_result('blur', "Blurred Image"))
sharpen_btn.on_click(lambda b: show_result('sharpen', "Sharpened Image"))
extra_blur_btn.on_click(lambda b: show_result('extra_blur', "Extra Blurred Image"))


# Display UI
display(
    uploader,  # upload button
    widgets.HBox([extra_blur_btn, blur_btn, sharpen_btn]),  # button row
    output  # output display area
)